# IndustryGPT — Specialized LLM Bot for Retail & E-commerce

**Capstone Project | Masters in Data Science**

**Project made by:** Santosh Kumar
**Industry chosen:** Retail and E-commerce
**GitHub Repository:** `<PASTE-YOUR-GITHUB-REPO-LINK-HERE>` &nbsp;*(see `industrygpt-retail-bot/README.md` for setup)*

---

## Project Summary

This project builds **IndustryGPT**, a domain-specific conversational AI assistant fine-tuned to answer
**Retail & E-commerce customer-support queries** — order tracking, returns & refunds, payments & billing,
product information, shipping & delivery, and account management.

A pre-trained sequence-to-sequence language model (**`google/flan-t5-small`**, sourced from Hugging Face)
is fine-tuned on a curated, industry-specific Q&A dataset (`retail_ecommerce_support.csv`, 120 labeled
question–answer pairs across 6 categories) using **Google Colab with a T4 GPU**, capped at a maximum of
**25 training epochs** as per project guidelines.

The notebook covers the full pipeline end-to-end:

1. Environment setup & dependency installation
2. Dataset loading and **exploratory data analysis (EDA)** with multiple charts
3. Data cleaning, preprocessing and train/validation split
4. Pre-trained model selection and tokenization
5. Fine-tuning with logged training/validation loss
6. Quantitative evaluation (ROUGE / BLEU, perplexity)
7. **Live interactive bot demo** — the required component for the video presentation
8. Conclusion, limitations, and future improvements

> **Note on reproducibility:** running this notebook top-to-bottom on a fresh Colab T4 runtime will
> regenerate every chart and metric from your own training run — the numbers you get may differ slightly
> from the report due to random initialization, which is expected and fine to discuss in your presentation.


## 1. Environment Setup
Install all required libraries. Run this cell first, in a fresh Colab runtime with **GPU (T4)** enabled: `Runtime → Change runtime type → T4 GPU`.

In [ ]:
# 1.1 Install dependencies (safe to re-run; pins avoid known breaking changes)
!pip install -q -U transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 \
    sentencepiece==0.2.0 evaluate==0.4.2 rouge_score==0.1.2 sacrebleu==2.4.3 \
    wordcloud==1.9.3 seaborn==0.13.2 pandas matplotlib scikit-learn
print("Dependencies installed successfully.")

In [ ]:
# 1.2 Imports
import os, re, random, textwrap
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import torch
from transformers import (
    T5Tokenizer, T5ForConditionalGeneration,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from datasets import Dataset
import evaluate

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE != "cuda":
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU for faster training.")

## 2. Load the Industry Dataset

The dataset `retail_ecommerce_support.csv` contains real-style Retail & E-commerce customer-support
Q&A pairs across 6 categories: **Order Tracking, Returns & Refunds, Payments & Billing, Product
Information, Shipping & Delivery, Account & Login**.

Upload `retail_ecommerce_support.csv` to your Colab session (left sidebar → Files → Upload), or mount
Google Drive and point `DATA_PATH` to it.

In [ ]:
# 2.1 Load data
DATA_PATH = "retail_ecommerce_support.csv"   # update path if using Google Drive

# Uncomment to mount Google Drive instead:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = "/content/drive/MyDrive/IndustryGPT/retail_ecommerce_support.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head(10)

In [ ]:
# 2.2 Basic cleaning
df = df.dropna(subset=["question", "answer"]).drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)
df["question"] = df["question"].str.strip()
df["answer"] = df["answer"].str.strip()
if "question_length" not in df.columns:
    df["question_length"] = df["question"].str.split().apply(len)
if "answer_length" not in df.columns:
    df["answer_length"] = df["answer"].str.split().apply(len)

print("After cleaning:", df.shape)
print("Categories:", df["category"].nunique())
df["category"].value_counts()

## 3. Exploratory Data Analysis (Charts)
Several charts to understand the dataset before training.

In [ ]:
# Chart 1: Category distribution (bar)
order = df["category"].value_counts()
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(x=order.values, y=order.index, hue=order.index, palette="viridis", legend=False, ax=ax)
ax.set_title("Chart 1 - Query Distribution by Category")
ax.set_xlabel("Number of Q&A pairs"); ax.set_ylabel("")
plt.show()

In [ ]:
# Chart 2: Category share (pie)
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(order.values, labels=order.index, autopct="%1.1f%%", colors=sns.color_palette("Set2", len(order)), startangle=90)
ax.set_title("Chart 2 - Category Share of Total Dataset")
plt.show()

In [ ]:
# Chart 3 & 4: Question / Answer length distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df["question_length"], bins=12, kde=True, color="#4C72B0", ax=axes[0])
axes[0].set_title("Chart 3 - Question Length (words)")
sns.histplot(df["answer_length"], bins=12, kde=True, color="#DD8452", ax=axes[1])
axes[1].set_title("Chart 4 - Answer Length (words)")
plt.tight_layout(); plt.show()

In [ ]:
# Chart 5: Question length by category (boxplot)
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=df, x="category", y="question_length", hue="category", palette="coolwarm", legend=False, ax=ax)
ax.set_title("Chart 5 - Question Length by Category")
ax.set_xlabel(""); ax.set_ylabel("Words")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.show()

In [ ]:
# Chart 6 & 7: Top keywords bar chart + word cloud
STOPWORDS = set(["how","do","i","is","the","my","a","an","to","for","of","what","are","can",
                  "on","in","this","it","if","and","or","you","your","does","will","be","get","was"])
text = " ".join(df["question"].str.lower())
words = [w for w in re.findall(r"[a-z']+", text) if w not in STOPWORDS and len(w) > 2]
common = Counter(words).most_common(15)

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x=[c for _, c in common], y=[w for w, _ in common], hue=[w for w, _ in common],
            palette="magma", legend=False, ax=ax)
ax.set_title("Chart 6 - Top 15 Frequent Keywords in Customer Queries")
ax.set_xlabel("Frequency")
plt.show()

wc = WordCloud(width=900, height=500, background_color="white", colormap="viridis").generate(" ".join(words))
fig, ax = plt.subplots(figsize=(9, 5))
ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
ax.set_title("Chart 7 - Word Cloud of Customer Queries")
plt.show()

## 4. Preprocessing & Train/Validation Split

Each row becomes an instruction-style input:
`"Answer the retail customer support question: <question>"` → target `<answer>`.
This instruction prefix helps the instruction-tuned `flan-t5-small` model apply its
general reasoning to this specific domain during fine-tuning.

In [ ]:
PREFIX = "Answer the retail customer support question: "

df["input_text"] = PREFIX + df["question"]
df["target_text"] = df["answer"]

dataset = Dataset.from_pandas(df[["input_text", "target_text", "category"]])
dataset = dataset.train_test_split(test_size=0.15, seed=SEED)
train_ds, val_ds = dataset["train"], dataset["test"]
print("Train size:", len(train_ds), "| Validation size:", len(val_ds))

## 5. Pre-trained Model Selection

**Model:** [`google/flan-t5-small`](https://huggingface.co/google/flan-t5-small) (~80M parameters).

Chosen because:
- It's an **instruction-tuned** encoder-decoder model, so it already follows natural-language
  instructions well before any fine-tuning — a strong starting point for a Q&A bot.
- Small enough to fine-tune comfortably on a free **Colab T4 GPU** within 25 epochs.
- Available directly from Hugging Face under an open license, satisfying the project's
  "use any pre-trained model from Hugging Face" requirement.

In [ ]:
MODEL_NAME = "google/flan-t5-small"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)

MAX_INPUT_LEN = 64
MAX_TARGET_LEN = 96

def preprocess(batch):
    model_inputs = tokenizer(batch["input_text"], max_length=MAX_INPUT_LEN, truncation=True, padding="max_length")
    labels = tokenizer(text_target=batch["target_text"], max_length=MAX_TARGET_LEN, truncation=True, padding="max_length")
    labels["input_ids"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in seq] for seq in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)
print("Tokenization complete.")

## 6. Fine-Tuning (max 25 epochs, T4 GPU)

Training uses Hugging Face's `Seq2SeqTrainer`. `EPOCHS` is capped at 25 per project guidelines —
lower it if you want a faster demo run (e.g. 8–10 epochs is usually enough to see the loss curve
flatten on this small dataset).

In [ ]:
EPOCHS = 15          # <= 25 as required by project guidelines
BATCH_SIZE = 8

training_args = Seq2SeqTrainingArguments(
    output_dir="./industrygpt-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=3e-4,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_strategy="epoch",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

train_result = trainer.train()
print("Training complete.")

In [ ]:
# Chart 8: Training vs Validation loss curve (from the trainer's real logs)
log_history = trainer.state.log_history
train_epochs, train_losses = [], []
eval_epochs, eval_losses = [], []
for entry in log_history:
    if "loss" in entry and "epoch" in entry and "eval_loss" not in entry:
        train_epochs.append(entry["epoch"]); train_losses.append(entry["loss"])
    if "eval_loss" in entry:
        eval_epochs.append(entry["epoch"]); eval_losses.append(entry["eval_loss"])

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(train_epochs, train_losses, marker="o", label="Training loss", color="#4C72B0")
ax.plot(eval_epochs, eval_losses, marker="s", label="Validation loss", color="#C44E52")
ax.set_title("Chart 8 - Fine-Tuning Loss Curve (Train vs Validation)")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend()
plt.show()

In [ ]:
# Chart 9: Learning rate schedule actually used during training
lr_epochs, lrs = [], []
for entry in log_history:
    if "learning_rate" in entry:
        lr_epochs.append(entry["epoch"]); lrs.append(entry["learning_rate"])

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(lr_epochs, lrs, marker="d", color="#55A868")
ax.set_title("Chart 9 - Learning Rate Schedule during Fine-Tuning")
ax.set_xlabel("Epoch"); ax.set_ylabel("Learning rate")
plt.show()

In [ ]:
# Chart 11: Validation perplexity across epochs (perplexity = exp(loss))
perplexities = [float(np.exp(l)) for l in eval_losses]
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(eval_epochs, perplexities, marker="o", color="#8172B2")
ax.set_title("Chart 11 - Validation Perplexity across Epochs")
ax.set_xlabel("Epoch"); ax.set_ylabel("Perplexity")
plt.show()

## 7. Quantitative Evaluation
Generate predictions on the held-out validation set and score them with ROUGE and BLEU.

In [ ]:
rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

model.eval()
preds, refs = [], []
for ex in val_ds:
    inp = tokenizer(ex["input_text"], return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(DEVICE)
    with torch.no_grad():
        out_ids = model.generate(**inp, max_length=MAX_TARGET_LEN, num_beams=4)
    pred = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    preds.append(pred)
    refs.append(ex["target_text"])

rouge_scores = rouge.compute(predictions=preds, references=refs)
bleu_score = bleu.compute(predictions=preds, references=[[r] for r in refs])

metrics = {
    "ROUGE-1": rouge_scores["rouge1"],
    "ROUGE-2": rouge_scores["rouge2"],
    "ROUGE-L": rouge_scores["rougeL"],
    "BLEU": bleu_score["score"] / 100,
}
print(metrics)

In [ ]:
# Chart 10: Evaluation metrics bar chart
fig, ax = plt.subplots(figsize=(6.5, 4.2))
sns.barplot(x=list(metrics.keys()), y=list(metrics.values()), hue=list(metrics.keys()),
            palette="crest", legend=False, ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Chart 10 - Bot Response Quality (Evaluation Metrics)")
ax.set_ylabel("Score")
for i, v in enumerate(metrics.values()):
    ax.text(i, v + 0.02, f"{v:.2f}", ha="center")
plt.show()

In [ ]:
# Chart 12: Sample predictions vs ground truth (qualitative check table)
sample_df = pd.DataFrame({
    "question": [ex["input_text"].replace("Answer the retail customer support question: ", "") for ex in val_ds][:8],
    "predicted_answer": preds[:8],
    "actual_answer": refs[:8],
})
pd.set_option("display.max_colwidth", 80)
sample_df

## 8. Live Bot Demo (REQUIRED for video presentation)

> ⚠️ **Per submission guidelines: you must interact with the bot LIVE during your recorded
> presentation.** Run the cell below and type real questions on camera — do not just show
> pre-written output.

In [ ]:
def ask_bot(question: str, max_length: int = MAX_TARGET_LEN, num_beams: int = 4) -> str:
    '''Query the fine-tuned IndustryGPT retail bot with a natural-language question.'''
    prompt = PREFIX + question.strip()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN).to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_length=max_length, num_beams=num_beams, early_stopping=True)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

# A few scripted example queries to sanity-check the bot
demo_questions = [
    "How can I track my order?",
    "What is your return policy?",
    "My payment failed but the amount was deducted, what now?",
    "Do you deliver internationally?",
    "How do I reset my password?",
]

for q in demo_questions:
    print("Q:", q)
    print("A:", ask_bot(q))
    print("-" * 80)

In [ ]:
# 8.1 Interactive loop - use this LIVE during your presentation.
# Type a question and press Enter. Type 'exit' to stop.
print("IndustryGPT Retail Support Bot — type 'exit' to quit.")
while True:
    user_q = input("You: ")
    if user_q.strip().lower() in ("exit", "quit"):
        print("Bot: Thank you for chatting with IndustryGPT. Goodbye!")
        break
    print("Bot:", ask_bot(user_q))

## 9. Save the Fine-Tuned Model
Save locally (and optionally to Google Drive) so it can be reloaded without retraining.

In [ ]:
SAVE_DIR = "./industrygpt-retail-flan-t5-small"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model and tokenizer saved to {SAVE_DIR}")

# Optional: copy to Google Drive
# import shutil
# shutil.copytree(SAVE_DIR, "/content/drive/MyDrive/IndustryGPT/model", dirs_exist_ok=True)

## 10. Conclusion, Limitations & Future Work

**What we built:** A retail/e-commerce customer-support chatbot by fine-tuning
`google/flan-t5-small` on a 120-example, 6-category, domain-specific Q&A dataset, trained on
a Colab T4 GPU within 25 epochs, with full EDA, training curves, and quantitative evaluation
(ROUGE / BLEU / perplexity).

**Challenges faced:**
- Building a clean, representative domain dataset from scratch (industry data collection).
- Balancing dataset size against overfitting risk given only 120 base examples.
- Occasional hallucinated or overly generic answers on out-of-distribution questions.

**Limitations:**
- Small training set limits generalization to questions far outside the 6 categories covered.
- No retrieval component — the bot cannot look up a specific customer's live order status.
- Evaluated on held-out questions from the same distribution as training data, not fully
  independent real-world traffic.

**Future improvements:**
- Add **Retrieval-Augmented Generation (RAG)** to ground answers in a live knowledge base /
  policy documents and real order data.
- Expand the dataset with real, anonymized customer-service transcripts.
- Add guardrails/safety filters for sensitive requests (e.g. payment disputes, PII).
- Human evaluation alongside automatic metrics (ROUGE/BLEU) for a fuller picture of quality.

**Connection to Industry Immersion module:** this project's findings on domain adaptation,
data collection strategy, and evaluation results feed directly into the follow-on research
paper for the Industry Immersion module.

---
**Project made by Santosh Kumar** | Masters in Data Science — Capstone Project
